In [ ]:
# ** This cell is needed since we are not in the src directory 
import sys 
import os
# Add the src/ directory to the Python path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../src/")))

ROOT_DIR = ".."
SRC_DIR = ROOT_DIR + "/src"

import sys
sys.path.append("/Users/admin/eeg-ds004504/")


In [ ]:
from config_handler import initiate_config, load_config

initiate_config()

In [ ]:
print(load_config())

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import pandas_udf, PandasUDFType
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, ArrayType, MapType
import pandas as pd

In [ ]:
# Check if there's an active Spark context and stop it
from pyspark import SparkContext
if SparkContext._active_spark_context:
    print("Stopping existing Spark context...")
    SparkContext._active_spark_context.stop()
    print("Previous Spark context stopped successfully")

In [ ]:
# creating the ultimate most optimised spark session ever muwahaha
import os
from pyspark.sql import SparkSession

# Set Java options for the JVM running Spark
# -Xmx12g : Sets the maximum heap size to 12GB
# -Xms4g : Sets the initial heap size to 4GB to avoid resizing overhead
os.environ["_JAVA_OPTIONS"] = "-Xmx16g -Xms8g"

# Build Spark session with memory, parallelism, and network settings
spark = (
    SparkSession.builder 
    # Application name shown in Spark UI
    .appName("EEG_Analysis") 

    # SPARK DIRECTORY FOR THREADS / PERSIST
    
    # Use all available logical cores or specify a number
    # "local[*]" uses all available cores, "local[12]" limits to 12 threads
    .config("spark.master", "local[12]") \

    # Executor memory: how much memory each Spark worker can use
    .config("spark.executor.memory", "8g") \

    # Driver memory: memory available to the Spark driver (main Python process)
    .config("spark.driver.memory", "8g") \
    
    # THIS IS WHERE SPARK WILL PUT ITS TEMPERARY VARIABLES 
    # .config("spark.local.dir", os.path.expanduser("~/external-spark-tmp"))

    # Number of shuffle partitions (e.g., after groupBy, join, etc.)
    # Lower this in local mode to reduce overhead (default is 200)
    .config("spark.sql.shuffle.partitions", "12") \

    # Default number of partitions in operations like parallelize
    .config("spark.default.parallelism", "12") \

    # Maximum size (in MB) allowed for any RPC message (e.g., large UDF closures or data broadcasts)
    .config("spark.rpc.message.maxSize", "512") \

    # prevent breaking pipes 
    .config("spark.reducer.maxReqsInFlight", "1") \
    .config("spark.shuffle.io.preferDirectBufs", "false") \
    .config("spark.shuffle.file.buffer", "32k") \
    
    # Required for avoiding binding issues on some MacOS environments
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.driver.host", "127.0.0.1") 
    .getOrCreate()
)

    # ----------------------------------------
    # Additional advanced options (optional):
    # ----------------------------------------

    # Use Kryo serializer instead of default Java serializer for better performance
    # .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \

    # Increase broadcast join timeout (in seconds) for large models or lookup tables
    # .config("spark.sql.broadcastTimeout", "600") \

    # Fraction of JVM memory reserved for execution and storage (default is 0.6)
    # .config("spark.memory.fraction", "0.8") \

    # Portion of memory reserved for caching/storage (default is 0.5 of memory.fraction)
    # .config("spark.memory.storageFraction", "0.3") \

    # Enable Apache Arrow for efficient pandas-to-Spark conversion (useful with UDFs)
    # .config("spark.sql.execution.arrow.pyspark.enabled", "true") \

    # Finalize and create the Spark session


# spark = SparkSession.builder.appName("MyApp").getOrCreate()

print("New Spark session created successfully")

In [ ]:
for item in spark.sparkContext.getConf().getAll():
    print(item)


In [ ]:
def reload_my_modules():
    import importlib
    import populate_schemas
    import feature_extraction
    import schema_definition
    importlib.reload(populate_schemas)
    importlib.reload(feature_extraction)
    importlib.reload(schema_definition)

reload_my_modules()


from populate_schemas import load_subjects_df, extract_features_udtf
from feature_extraction import processEpoch, processSub
from schema_definition import get_feature_schema, get_subject_schema

sc = spark.sparkContext # we pass udf/udtf's (user defind functions and user defined table functions) to spark so it can access them

# Making all necessary modules available to spark
try: 
    # oh btw spark is werid about not finding the config but it always finds it somehow not sure how that works not going to look rn tbh
    # ^ so feature_extraction has extra print statements
    sc.addPyFile(os.path.join(SRC_DIR, "feature_extraction.py"))
    print("Added feature_extraction.py to the pyspark context")
    sc.addPyFile(os.path.join(SRC_DIR, "preprocess_sets.py"))
    print("Added preprocess_sets to the pyspark context")
    sc.addPyFile(os.path.join(SRC_DIR, "schema_definition.py"))
    print("Added schema_definition.py to the pyspark context")
    sc.addPyFile(os.path.join(SRC_DIR, "config_handler.py"))
    print("Added config_handler.py to the pyspark context")
except Exception as e:
    print(f"Error adding files to SparkContext: {e}")

In [ ]:
import pandas as pd

# alz_df_pandas = pd.read_pickle("alz_df_apr10_1355.pkl")
# cntrl_df_pandas = pd.read_pickle("cntrl_df_apr10_1355.pkl")


# alz_df_pandas = pd.read_pickle("features_alz_extra_features_Apr15_1033.pkl")
# cntrl_df_pandas = pd.read_pickle("features_cntrl_extra_features_Apr15_1033.pkl")

alz_df_pandas = pd.read_pickle("features_alz_extra_features_Apr19_2141.pkl")
cntrl_df_pandas = pd.read_pickle("features_cntrl_extra_features_Apr19_2141.pkl")


In [ ]:
%%time
alz_df_spark = spark.createDataFrame(alz_df_pandas)
cntrl_df_spark = spark.createDataFrame(cntrl_df_pandas)



In [ ]:
%%time
alz_df_spark = spark.read.parquet("features_alz_extra_features_Apr19_2141.parquet")
cntrl_df_spark = spark.read.parquet("features_cntrl_extra_features_Apr19_2141.parquet")

In [ ]:
# alz_df_spark.parquet("spark_features_alz_extra_features_Apr19_2141.parquet")
# cntrl_df_spark.parquet("spark_features_cntrl_extra_features_Apr19_2141.parquet")

In [ ]:
#just renaming things now that we understand the types and where things are coming from
alz_df = alz_df_spark
cntrl_df = cntrl_df_spark

In [ ]:
alz_df.show()

# Raw Data Visualization

# Start of data processing

In [ ]:
# give each its respective labels
from pyspark.sql.functions import lit
alz_df = alz_df.withColumn("label", lit(1)).repartition(8).persist()
cntrl_df = cntrl_df.withColumn("label", lit(0)).repartition(8).persist()

In [ ]:
# union everything
full_df = alz_df.unionByName(cntrl_df)

In [ ]:
# Split based on feature type
from pyspark.sql.functions import col

band_df = full_df.filter(col("table_type") == "band")
channel_df = full_df.filter(col("table_type") == "electrode")
epoch_df = full_df.filter(col("table_type") == "epoch")


In [ ]:
alz_df.unpersist()
cntrl_df.unpersist()

In [ ]:
from pyspark.sql.functions import col

# Filter and save each to Parquet
full_df.filter(col("table_type") == "band") \
    .write.mode("overwrite").parquet("tempParquets/band_df")

full_df.filter(col("table_type") == "electrode") \
    .write.mode("overwrite").parquet("tempParquets/channel_df")

full_df.filter(col("table_type") == "epoch") \
    .write.mode("overwrite").parquet("tempParquets/epoch_df")

In [ ]:
import os
os.system('say "START!"')

In [ ]:
from pyspark.sql.functions import first
from pyspark.sql.functions import concat_ws


# Band-level: Electrode_WaveBand_Feature
band_df = band_df.withColumn("pivot", concat_ws("_", "Electrode", "WaveBand", "FeatureName")).repartition(1000, "SubjectID").persist()

# band_df.write.mode("overwrite").parquet("band_pre_pivot")
# Pivot band-level features
band_pivot = band_df.groupBy("SubjectID", "EpochID", "label").pivot("pivot").agg(first("FeatureValue"))
band_df.unpersist()

# Channel-level: Electrode_Feature
channel_df = channel_df.withColumn("pivot", concat_ws("_", "Electrode", "FeatureName")).repartition(1000, "SubjectID").persist()
# channel_df.write.mode("overwrite").parquet("channel_df_pre_pivot")

# Pivot channel-level features
channel_pivot = channel_df.groupBy("SubjectID", "EpochID", "label").pivot("pivot").agg(first("FeatureValue"))
channel_pivot.unpersist()


# Band-level: Electrode_WaveBand_Feature
epoch_df = epoch_df.withColumn("pivot", col("FeatureName")).repartition(1000, "SubjectID").persist()
# epoch_df.write.mode("overwrite").parquet("epoch_df_pre_pivot")


# Pivot epoch-level features
epoch_pivot = epoch_df.groupBy("SubjectID", "EpochID", "label").pivot("pivot").agg(first("FeatureValue"))
epoch_pivot.unpersist()

In [ ]:
import os
os.system('say "DONE!"')

In [ ]:
from pyspark.sql.functions import first

# Pivot band-level features
band_pivot = band_df.groupBy("SubjectID", "EpochID", "label").pivot("pivot").agg(first("FeatureValue"))

# Pivot channel-level features
channel_pivot = channel_df.groupBy("SubjectID", "EpochID", "label").pivot("pivot").agg(first("FeatureValue"))

# Pivot epoch-level features
epoch_pivot = epoch_df.groupBy("SubjectID", "EpochID", "label").pivot("pivot").agg(first("FeatureValue"))

In [ ]:
from functools import reduce

# full_df = reduce(
#     lambda df1, df2: df1.join(df2, on=["SubjectID", "EpochID", "label"], how="outer"),
#     [band_pivot, channel_pivot, epoch_pivot]
# ).fillna(0.0)


full_df = reduce(
    lambda df1, df2: df1.join(df2, on=["SubjectID", "EpochID", "label"], how="outer"),
     [band_pivot, channel_pivot, epoch_pivot]# band_pivot, channel_pivot, epoch_pivot]
).fillna(0.0)
full_df.repartition(16).persist()


In [ ]:
band_pivot.unpersist()
channel_pivot.unpersist()
epoch_pivot.unpersist()

In [ ]:
type(full_df)

In [ ]:
full_df.repartition(16).persist()


In [ ]:
full_df.write.mode("overwrite").parquet("tempParquets/full_df")

In [ ]:
print("w")

In [ ]:
NUM_TEST_SUBJECTS_PER_GROUP = 2

# Get test subject IDs from full_df (which has .label)
alz_test_subjects = (
    full_df.filter("label == 1")
    .select("SubjectID")
    .distinct()
    .orderBy("SubjectID")
    .limit(NUM_TEST_SUBJECTS_PER_GROUP)
    .rdd.flatMap(lambda row: row)
    .collect()
)

cntrl_test_subjects = (
    full_df.filter("label == 0")
    .select("SubjectID")
    .distinct()
    .orderBy("SubjectID")
    .limit(NUM_TEST_SUBJECTS_PER_GROUP)
    .rdd.flatMap(lambda row: row)
    .collect()
)

test_subjects = alz_test_subjects + cntrl_test_subjects #this will be the firs 2 subjects of each group for reproduceablility

In [ ]:
print("w")

In [ ]:
# from pyspark.sql.functions import rand

# NUM_TEST_SUBJECTS_PER_GROUP = 2
# SEED = 42

# # Alzheimer's test subjects (label == 1)
# alz_test_subjects = (
#     full_df.filter("label == 1")
#     .select("SubjectID")
#     .distinct()
#     .orderBy(rand(SEED))  # Randomize with seed
#     .limit(NUM_TEST_SUBJECTS_PER_GROUP)
#     .rdd.flatMap(lambda row: row)
#     .collect()
# )

# # Control test subjects (label == 0)
# cntrl_test_subjects = (
#     full_df.filter("label == 0")
#     .select("SubjectID")
#     .distinct()
#     .orderBy(rand(SEED + 1))  # Different seed for different shuffle
#     .limit(NUM_TEST_SUBJECTS_PER_GROUP)
#     .rdd.flatMap(lambda row: row)
#     .collect()
# )

# # Combine test subjects
# test_subjects = alz_test_subjects + cntrl_test_subjects


In [ ]:
print(f"alz_test_subjects {alz_test_subjects}")
print(f"cntrl_test_subjects {cntrl_test_subjects}")

In [ ]:
# Split into test and train sets
train_df = full_df.filter(~col("SubjectID").isin(test_subjects))
test_df = full_df.filter(col("SubjectID").isin(test_subjects))


In [ ]:
print("got here") 

# DO T-TEST HERE !!

In [ ]:
train_df.columns

In [ ]:
# import dimensionality_reduction
# import importlib
# importlib.reload(dimensionality_reduction)
# from dimensionality_reduction import min_max_normalize, normalize_by_column_per_subject_wide
# feature_cols = [c for c in train_df.columns if c not in ("SubjectID", "EpochID", "label")]


# # train_norm_df, test_norm_df = min_max_normalize(train_df, test_df, feature_cols)
# train_norm_df = normalize_by_column_per_subject_wide(train_df, feature_cols)
# test_norm_df = normalize_by_column_per_subject_wide(test_df, feature_cols)

In [ ]:
train_df.head(1)

# NORMALIZING SUJBECT WIDE FOR EXPERIMENT< REMOVE!, MIN MAX< TRY DIFFERNET COMBOS FOR LOWER COHEN!

In [ ]:
import importlib
try:
    importlib.reload(dimensionality_reduction)
except:
    pass
from dimensionality_reduction import min_max_normalize, normalize_by_column, normalize_by_column_per_subject_wide

# normalize_by_column_per_subject_wide(

In [ ]:
# train_df.schema

In [ ]:
# import dimensionality_reduction
import importlib
try:
    importlib.reload(dimensionality_reduction)
except:
    pass
from dimensionality_reduction import min_max_normalize, normalize_by_column, normalize_by_column_per_subject_wide
feature_cols = [c for c in train_df.columns if c not in ("SubjectID", "EpochID", "label")]

# train_norm_df, test_norm_df = train_df, test_df

# train_norm_df = normalize_by_column_per_subject_wide(train_df, feature_cols)
# test_norm_df = normalize_by_column_per_subject_wide(test_df, feature_cols)

# RUNNING T-TEST - using regression to do the test like named here 
# https://stackoverflow.com/questions/58851008/how-to-perform-student-t-test-in-pyspark

from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression

results = []
for feat in feature_cols:
    # Step 1: Assemble the single feature into featuresCol
    assembler = VectorAssembler(inputCols=["label"], outputCol="features")
    assembled = assembler.transform(train_df.select("label", feat).dropna())

    # Step 2: Fit regression model: feature ~ label
    lr = LinearRegression(featuresCol="features", labelCol=feat, regParam=0)
    model = lr.fit(assembled)

    # Step 3: Get t-stat and p-value for 'label'
    summary = model.summary
    t_stat = summary.tValues[1]  # index 1 corresponds to label coefficient
    p_val = summary.pValues[1]

    # Save results
    results.append((feat, t_stat, p_val))




results_df = pd.DataFrame(results, columns=["Feature", "T_statistic", "P_value"])
results_df.sort_values("P_value", inplace=True)  # sort by significance


# add benferroni or FDR correction 
# from statsmodels.stats.multitest import multipletests

# # Apply FDR correction
# rejected, pvals_corrected, _, _ = multipletests(results_df["P_value"], alpha=0.05, method="fdr_bh")
# results_df["FDR_corrected"] = pvals_corrected
# results_df["Significant"] = rejected


# train_norm_df.repartition(16).persist()
# test_norm_df.repartition(16).persist()
print("finished T-test and have results")

In [ ]:
# train_norm_df.head(1)
results_df


In [ ]:
results_df = results_df.sort_values(by="T_statistic", ascending=False)

In [ ]:
results_df

In [ ]:
# results_df.to_pickle("ttest_results.pkl")


In [ ]:
from pyspark.sql.functions import col, avg, stddev, count

cohen_d_results = []

# remember 1 is alzeimers and 0 is control, so if negative cohen, means control's mean was larger 
for feat in feature_cols:
    stats = (
        train_df
        .select("label", feat)
        .dropna()
        .groupBy("label")
        .agg(
            avg(feat).alias("mean"),
            stddev(feat).alias("std"),
            count(feat).alias("n")
        )
        .toPandas()
        .set_index("label")
    )
    
    if 0 in stats.index and 1 in stats.index:
        mean0 = stats.loc[0, "mean"]
        mean1 = stats.loc[1, "mean"]
        std0 = stats.loc[0, "std"]
        std1 = stats.loc[1, "std"]
        n0 = stats.loc[0, "n"]
        n1 = stats.loc[1, "n"]

        # pooled std
        pooled_std = (( (n0 - 1) * std0**2 + (n1 - 1) * std1**2 ) / (n0 + n1 - 2)) ** 0.5
        cohen_d = (mean1 - mean0) / pooled_std if pooled_std > 0 else 0.0
    else:
        cohen_d = None

    cohen_d_results.append((feat, cohen_d))


In [ ]:
# cohen_d_results.to_csv("cohen_results.csv")

In [ ]:
cohen_d_results

In [ ]:
import os
os.system('say "t-test is done!"')

In [ ]:
cohen_df = pd.DataFrame(cohen_d_results, columns=["Feature", "Cohen_d"])
results_df = results_df.merge(cohen_df, on="Feature")
results_df["Abs_Cohen_d"] = results_df["Cohen_d"].abs()

In [ ]:
results_df = results_df.sort_values(by="Abs_Cohen_d", ascending=False)

In [ ]:
results_df[results_df["Abs_Cohen_d"] > 0.2].count()

In [ ]:
results_df[results_df["Abs_Cohen_d"] > 0.2]

In [ ]:
results_df.to_pickle("ttest+cohen_results_FILTERED.pkl")
results_df.to_csv("ttest+cohen_results_FILTERED.csv")

# Dimensionality reduction with t-test

In [ ]:
# import importlib
# try:
#     importlib.reload(dimensionality_reduction)
# except:
#     pass
# from dimensionality_reduction import apply_pca_model

# train_df = apply_pca_model(train_norm_df, pca_input_cols, pca_model, k_val)
# test_df = apply_pca_model(test_norm_df, pca_input_cols, pca_model, k_val)


In [ ]:
results_df = pd.read_pickle("ttest+cohen_results.pkl")

In [ ]:
cohen_min = 0.5
features_of_interest = results_df[results_df["Abs_Cohen_d"] > cohen_min]["Feature"].tolist()

In [ ]:
print(f"After cohen test we have {len( features_of_interest)} features of interest with a cohen value greater then {cohen_min}")
print(features_of_interest)

In [ ]:
# Required columns to retain
meta_cols = ["label", "SubjectID", "EpochID"]  # add/remove as needed
selected_cols = meta_cols + features_of_interest
features_of_interest


In [ ]:
train_df.head(1)

In [ ]:
train_df = train_df.select(*selected_cols)
test_df = test_df.select(*selected_cols)

In [ ]:
train_df.head(1)

In [ ]:
len(train_df.head(1)[0])

In [ ]:
# Min maxing accross everything

In [ ]:
from pyspark.sql.functions import min as spark_min, max as spark_max

# Example list of selected features (from Cohen's d filtering)
# You already have: features_of_interest
stat_exprs = []

for feature in features_of_interest:
    stat_exprs.append(spark_min(feature).alias(f"{feature}_min"))
    stat_exprs.append(spark_max(feature).alias(f"{feature}_max"))

# Compute min and max for all selected features
feature_ranges = train_df.select(*features_of_interest).agg(*stat_exprs)



# we need to try z-score, z-score by subjet, min_max, min_max by subject

In [ ]:
from pyspark.sql.functions import col, lit

feature_ranges_row = feature_ranges.collect()[0].asDict()

for feat in features_of_interest:
    min_val = feature_ranges_row[f"{feat}_min"]
    max_val = feature_ranges_row[f"{feat}_max"]
    denom = max_val - min_val if max_val != min_val else 1.0  # avoid divide-by-zero

    # Overwrite the original column
    train_df = train_df.withColumn(
        feat,
        (2 * (col(feat) - lit(min_val)) / lit(denom)) - 1
    )

    test_df = test_df.withColumn(
        feat,
        (2 * (col(feat) - lit(min_val)) / lit(denom)) - 1
    )


In [ ]:
train_df.head()

In [ ]:
from pyspark.sql.functions import min as spark_min, max as spark_max

# Example list of selected features (from Cohen's d filtering)
# You already have: features_of_interest
stat_exprs = []

for feature in features_of_interest:
    stat_exprs.append(spark_min(feature).alias(f"{feature}_min"))
    stat_exprs.append(spark_max(feature).alias(f"{feature}_max"))

# Compute min and max for all selected features
feature_ranges = train_df.select(*features_of_interest).agg(*stat_exprs)

In [ ]:
feature_ranges.head()

In [ ]:
import importlib
try:
    importlib.reload(dimensionality_reduction)
except:
    pass
    
from dimensionality_reduction import min_max_normalize, normalize_by_column, normalize_by_column_per_subject_wide

# train_df = normalize_by_column_per_subject_wide(train_df, feature_cols)
# test_df = normalize_by_column_per_subject_wide(test_df, feature_cols)


In [ ]:
print("got here")

# ML time

In [ ]:
train_df.columns

In [ ]:
train_pd = train_df.toPandas()
test_pd = test_df.toPandas()
# spark.stop()

In [ ]:
test_pd.columns.tolist()

In [ ]:
train_pd.columns.tolist()

In [ ]:
import numpy as np

# Convert Spark DenseVectors to regular 2D numpy arrays
# X_train = np.array(train_pd["features"].tolist())
# y_train = train_pd["label"].values

# X_test = np.array(test_pd["features"].tolist())
# y_test = test_pd["label"].values

exclude_cols = ["label", "SubjectID", "EpochID"]
feature_cols = [col for col in train_pd.columns if col not in exclude_cols]

# Features matrix
X_train = train_pd[feature_cols].values
X_test = test_pd[feature_cols].values

# Labels
y_train = train_pd["label"].values
y_test = test_pd["label"].values


In [ ]:
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)


In [ ]:
y_train

In [ ]:
X_train[0]

In [ ]:
len(X_train[0])

In [ ]:
# How can we do standard scaler per subject ! !!!  ! ! ! !  !  !! ! !  ! 

In [ ]:
print("work")

In [ ]:
from sklearn.preprocessing import StandardScaler


X_train_scaled = X_train

X_test_scaled = X_test

# making sure min-maxed ! also might change results a little 

# scaler = StandardScaler()

# X_train_scaled = scaler.fit_transform(X_train)

# X_test_scaled = scaler.transform(X_test)

In [ ]:
print("here")

In [ ]:
%%time
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler
import numpy as np
import time

# # Step 1: Scale once
# scaler = StandardScaler()
# X_train_scaled = scaler.fit_transform(X_train)
# X_test_scaled = scaler.transform(X_test)

# Step 2: Define hyperparameter grid
# k_values = [1, 2, 3, 5, 7, 9, 11]
k_values = [7, 9, 11, 13, 15, 17, 20]
weights_list = ['uniform', 'distance']
metrics = ['euclidean', 'manhattan']
p_values = [1, 2]  # Only used if metric is 'minkowski'

# Step 3: Set up result tracking
results = []
target_names = ["Control", "Alzheimer's"]
print("start")

# Step 4: Manual hyperparameter search
for k in k_values:
    for weight in weights_list:
        for metric in metrics:
            for p in p_values:

                if metric != 'minkowski' and p != 2:
                    continue  # p is irrelevant unless using 'minkowski'

                label = f"KNN k={k}, weight={weight}, metric={metric}, p={p}"
                model = KNeighborsClassifier(
                    n_neighbors=k,
                    weights=weight,
                    metric=metric if metric != 'minkowski' else 'minkowski',
                    p=p
                )

                print(f"\n=== Cross-Validation: {label} ===")
                start = time.time()

                # Step 5: Cross-validation scores
                scores = cross_val_score(model, X_train_scaled, y_train, cv=15, scoring='accuracy', n_jobs=3)
                mean_acc, std_acc = scores.mean(), scores.std()
                print(f"Mean Accuracy: {mean_acc:.4f}")
                print(f"Std Deviation: {std_acc:.4f}")
                print(f"All Fold Scores: {np.round(scores, 4)}")

                # Step 6: Best fold evaluation
                skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
                best_fold_index = np.argmax(scores)
                for i, (train_idx, test_idx) in enumerate(skf.split(X_train_scaled, y_train)):
                    if i == best_fold_index:
                        X_tr, X_te = X_train_scaled[train_idx], X_train_scaled[test_idx]
                        y_tr, y_te = y_train[train_idx], y_train[test_idx]

                        model.fit(X_tr, y_tr)
                        y_pred_test = model.predict(X_te)
                        y_pred_train = model.predict(X_tr)

                        train_acc = accuracy_score(y_tr, y_pred_train)
                        val_acc = accuracy_score(y_te, y_pred_test)

                        print(f"\n=== Best Fold Summary: {label} ===")
                        print(f"Train Accuracy: {train_acc:.4f}")
                        print(f"Validation Accuracy: {val_acc:.4f}")
                        print(classification_report(y_te, y_pred_test, target_names=target_names))

                    # Step 7: Evaluate on the held-out test set
                    #!! shouldn't we be testing oin the model that is trained on all the folds 1 by 1 (so multiple epochs) ?
                    model.fit(X_train_scaled, y_train)
                    y_test_pred = model.predict(X_test_scaled)
                    test_acc = accuracy_score(y_test, y_test_pred)
                    print(f"Test Accuracy: {test_acc:.4f}")
                    print(classification_report(y_test, y_test_pred, target_names=target_names))

                    results.append((label, mean_acc, std_acc, train_acc, val_acc, test_acc))
                    break

                print(f"⏱️ Duration: {time.time() - start:.1f}s")

# Step 8: Final summary sorted by CV accuracy
results.sort(key=lambda x: x[1], reverse=True)
print("\n=== Top Models by Mean CV Accuracy ===")
for label, mean_acc, std_acc, train_acc, val_acc, test_acc in results:
    print(f"{label:<65} -> CV: {mean_acc:.4f} ± {std_acc:.4f} | Train: {train_acc:.4f} | Validation: {val_acc:.4f} | Test: {test_acc:.4f}")


In [ ]:
%%time
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import MinMaxScaler
import numpy as np
import time

# Step 1: Scale data once (IMPORTANT: transform X_test with same scaler)
# scaler = MinMaxScaler(feature_range=(-1, 1))
# X_train_scaled = scaler.fit_transform(X_train)
# X_test_scaled = scaler.transform(X_test)

# Step 2: Hyperparameters
layer_configs = [(256, 128, 64), (128, 64, 16)]
activations = ['relu']
alphas = [1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 5e-1, 1.0]
early_stopping_options = [True] #, False] # !! REMOVED FALSE, TOOK TOO LONG
max_iter = 30000

# Step 3: Set up result tracking
results = []
target_names = ["Control", "Alzheimer's"]

# Step 4: Brute-force hyperparameter loop
for layers in layer_configs:
    for activation in activations:
        for alpha in alphas:
            for early_stopping in early_stopping_options:

                label = f"MLP {layers}, act={activation}, alpha={alpha}, early_stop={early_stopping}, max_iter={max_iter}"
                model = MLPClassifier(
                    hidden_layer_sizes=layers,
                    activation=activation,
                    alpha=alpha,
                    early_stopping=early_stopping,
                    max_iter=max_iter,
                    random_state=42,
                )

                print(f"\n=== Cross-Validation: {label} ===")
                start = time.time()

                # Step 5: Cross-validation
                scores = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring='accuracy', n_jobs=3)
                mean_acc, std_acc = scores.mean(), scores.std()
                print(f"Mean Accuracy: {mean_acc:.4f}")
                print(f"Std Deviation: {std_acc:.4f}")
                print(f"All Fold Scores: {np.round(scores, 4)}")

                # Step 6: Evaluate on best validation fold
                skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
                best_fold_index = np.argmax(scores)
                for i, (train_idx, test_idx) in enumerate(skf.split(X_train_scaled, y_train)):
                    if i == best_fold_index:
                        X_tr, X_te = X_train_scaled[train_idx], X_train_scaled[test_idx]
                        y_tr, y_te = y_train[train_idx], y_train[test_idx]

                        model.fit(X_tr, y_tr)
                        y_pred_test = model.predict(X_te)
                        y_pred_train = model.predict(X_tr)

                        train_acc = accuracy_score(y_tr, y_pred_train)
                        val_acc = accuracy_score(y_te, y_pred_test)

                        print(f"\n=== Best Fold Summary: {label} ===")
                        print(f"Train Accuracy: {train_acc:.4f}")
                        print(f"Validation Accuracy: {val_acc:.4f}")
                        print(classification_report(y_te, y_pred_test, target_names=target_names))

                        break

                # Step 7: Evaluate on held-out test set
                model.fit(X_train_scaled, y_train)
                y_test_pred = model.predict(X_test_scaled)
                test_acc = accuracy_score(y_test, y_test_pred)
                print(f"\n=== Final Test Set Evaluation: {label} ===")
                print(f"Test Accuracy: {test_acc:.4f}")
                print(classification_report(y_test, y_test_pred, target_names=target_names))

                results.append((label, mean_acc, std_acc, train_acc, val_acc, test_acc))

                print(f"⏱️ Duration: {time.time() - start:.1f}s")

# Step 8: Print final summary
results.sort(key=lambda x: x[1], reverse=True)
print("\n=== Top Models by Mean CV Accuracy ===")
for label, mean_acc, std_acc, train_acc, val_acc, test_acc in results:
    print(f"{label:<75} -> CV: {mean_acc:.4f} ± {std_acc:.4f} | Train: {train_acc:.4f} | Validation: {val_acc:.4f} | Test: {test_acc:.4f}")


In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report
import numpy as np
import time


n_estimators_list = [100, 200, 300, 400]
n_estimators_list.sort(reverse=True)
learning_rates = [0.1, 0.05, 0.01]
learning_rates.sort(reverse=True)  
max_depths = [3, 5, 7, 9]#  12]
max_depths.sort(reverse=True)         
subsample_rates = [0.6, 0.8, 1.0]   




# Result tracker
results = []
target_names = ["Control", "Alzheimer's"]

# Brute-force sweep
for n_estimators in n_estimators_list:
    for learning_rate in learning_rates:
        if learning_rate == 0.05 and n_estimators < 200:
            continue  # skip inefficient combos

        for max_depth in max_depths:
            for subsample in subsample_rates:

                label = (f"GBT n_estimators={n_estimators}, lr={learning_rate}, "
                         f"depth={max_depth}, subsample={subsample}")
                model = GradientBoostingClassifier(
                    n_estimators=n_estimators,
                    learning_rate=learning_rate,
                    max_depth=max_depth,
                    subsample=subsample,
                    random_state=42
                )

                print(f"\n🌲 Cross-Validation: {label}")
                start = time.time()

                scores = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring='accuracy', n_jobs=3)
                mean_acc, std_acc = scores.mean(), scores.std()
                print(f"Mean Accuracy: {mean_acc:.4f}, Std Dev: {std_acc:.4f}")
                print(f"Fold Scores: {np.round(scores, 4)}")

                # Best fold eval
                skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
                best_fold_index = np.argmax(scores)
                for i, (train_idx, val_idx) in enumerate(skf.split(X_train_scaled, y_train)):
                    if i == best_fold_index:
                        X_tr, X_val = X_train_scaled[train_idx], X_train_scaled[val_idx]
                        y_tr, y_val = y_train[train_idx], y_train[val_idx]

                        model.fit(X_tr, y_tr)
                        y_val_pred = model.predict(X_val)
                        y_tr_pred = model.predict(X_tr)

                        train_acc = accuracy_score(y_tr, y_tr_pred)
                        val_acc = accuracy_score(y_val, y_val_pred)

                        print(f"\n✅ Best Fold Summary: {label}")
                        print(f"Train Accuracy:      {train_acc:.4f}")
                        print(f"Validation Accuracy: {val_acc:.4f}")
                        print(classification_report(y_val, y_val_pred, target_names=target_names))
                        break

                # Test set eval
                model.fit(X_train_scaled, y_train)
                y_test_pred = model.predict(X_test_scaled)
                test_acc = accuracy_score(y_test, y_test_pred)

                print(f"\n🧪 Test Set Evaluation: {label}")
                print(f"Test Accuracy: {test_acc:.4f}")
                print(classification_report(y_test, y_test_pred, target_names=target_names))

                results.append((label, mean_acc, std_acc, train_acc, val_acc, test_acc))
                print(f"⏱️ Duration: {time.time() - start:.1f}s")

# Sort and show top configs
results.sort(key=lambda x: x[1], reverse=True)
print("\n=== 🏆 Top Gradient Boosting Models by CV Accuracy ===")
for label, mean_acc, std_acc, train_acc, val_acc, test_acc in results:
    print(f"{label:<75} -> CV: {mean_acc:.4f} ± {std_acc:.4f} | Train: {train_acc:.4f} | Val: {val_acc:.4f} | Test: {test_acc:.4f}")


In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import MinMaxScaler
import numpy as np
import time


layer_configs = [(256, 128, 64)]
activations = ['relu']
alphas = [1e-4]
early_stopping_options = [True]

results = []
target_names = ["Control", "Alzheimer's"]

for layers in layer_configs:
    for activation in activations:
        for alpha in alphas:
            for early_stopping in early_stopping_options:

                label = f"MLP {layers}, act={activation}, alpha={alpha}, early_stop={early_stopping}"
                model = MLPClassifier(
                    hidden_layer_sizes=layers,
                    activation=activation,
                    alpha=alpha,
                    early_stopping=early_stopping,
                    max_iter=10000,
                    random_state=42,
                )

                print(f"\n=== Cross-Validation: {label} ===")
                start = time.time()

                # Step 5: Cross-validation accuracy scores
                scores = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring='accuracy', n_jobs=3)
                mean_acc, std_acc = scores.mean(), scores.std()
                print(f"Mean Accuracy: {mean_acc:.4f}")
                print(f"Std Deviation: {std_acc:.4f}")
                print(f"All Fold Scores: {np.round(scores, 4)}")

                # Step 6: Find best fold for detailed summary
                skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
                best_fold_index = np.argmax(scores)
                for i, (train_idx, test_idx) in enumerate(skf.split(X_train_scaled, y_train)):
                    if i == best_fold_index:
                        X_tr, X_te = X_train_scaled[train_idx], X_train_scaled[test_idx]
                        y_tr, y_te = y_train[train_idx], y_train[test_idx]

                        model.fit(X_tr, y_tr)
                        y_pred_test = model.predict(X_te)
                        y_pred_train = model.predict(X_tr)

                        train_acc = accuracy_score(y_tr, y_pred_train)
                        test_acc = accuracy_score(y_te, y_pred_test)

                        print(f"\n=== Best Fold Summary: {label} ===")
                        print(f"Train Accuracy: {train_acc:.4f}")
                        print(f"Test Accuracy: {test_acc:.4f}")
                        print(classification_report(y_te, y_pred_test, target_names=target_names))

                        results.append((label, mean_acc, std_acc, train_acc, test_acc))
                        break

                print(f"⏱️ Duration: {time.time() - start:.1f}s")


results.sort(key=lambda x: x[1], reverse=True)
print("\n=== Top Models by Mean CV Accuracy ===")
for label, mean_acc, std_acc, train_acc, test_acc in results:
    print(f"{label:<65} -> CV: {mean_acc:.4f} ± {std_acc:.4f} | Train: {train_acc:.4f} | Test: {test_acc:.4f}")


In [ ]:
%%time
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report
import numpy as np
import time

X_train_boost = X_train_scaled  # assuming you already have this
X_test_boost = X_test_scaled

n_estimators_list =  [200]
learning_rates = [0.1]
max_depths = [3]
min_samples_leafs = [1]

results = []
target_names = ["Control", "Alzheimer's"]

for n_est in n_estimators_list:
    for lr in learning_rates:
        for depth in max_depths:
            for min_leaf in min_samples_leafs:

                label = f"GradBoost n={n_est}, lr={lr}, depth={depth}, min_leaf={min_leaf}"
                model = GradientBoostingClassifier(
                    n_estimators=n_est,
                    learning_rate=lr,
                    max_depth=depth,
                    min_samples_leaf=min_leaf,
                    random_state=42
                )

                print(f"\n=== Cross-Validation: {label} ===")
                start = time.time()

                # Step 5: Cross-validation scores
                scores = cross_val_score(model, X_train_boost, y_train, cv=5, scoring='accuracy', n_jobs=3)
                mean_acc, std_acc = scores.mean(), scores.std()
                print(f"Mean Accuracy: {mean_acc:.4f}")
                print(f"Std Deviation: {std_acc:.4f}")
                print(f"All Fold Scores: {np.round(scores, 4)}")

                # Step 6: Best fold evaluation
                skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
                best_fold_index = np.argmax(scores)
                for i, (train_idx, test_idx) in enumerate(skf.split(X_train_boost, y_train)):
                    if i == best_fold_index:
                        X_tr, X_te = X_train_boost[train_idx], X_train_boost[test_idx]
                        y_tr, y_te = y_train[train_idx], y_train[test_idx]

                        model.fit(X_tr, y_tr)
                        y_pred_test = model.predict(X_te)
                        y_pred_train = model.predict(X_tr)

                        train_acc = accuracy_score(y_tr, y_pred_train)
                        test_acc = accuracy_score(y_te, y_pred_test)

                        print(f"\n=== Best Fold Summary: {label} ===")
                        print(f"Train Accuracy: {train_acc:.4f}")
                        print(f"Test Accuracy: {test_acc:.4f}")
                        print(classification_report(y_te, y_pred_test, target_names=target_names))

                        results.append((label, mean_acc, std_acc, train_acc, test_acc))
                        break

                print(f"⏱️ Duration: {time.time() - start:.1f}s")

results.sort(key=lambda x: x[1], reverse=True)
print("\n=== Top Models by Mean CV Accuracy ===")
for label, mean_acc, std_acc, train_acc, test_acc in results:
    print(f"{label:<75} -> CV: {mean_acc:.4f} ± {std_acc:.4f} | Train: {train_acc:.4f} | Test: {test_acc:.4f}")
